# KMeansClassifier


In [1]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler


In [2]:
# CONFIG
input_csv = "AllCars.csv"
number_of_clusters = 5
random_state = 42


In [3]:
# Load data
dataset = pd.read_csv(input_csv)

required = {"Volume","Doors","Style"}
missing = required - set(dataset.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

X = dataset[["Volume","Doors"]].copy()
X = X.apply(pd.to_numeric, errors="coerce")
med = X.median(numeric_only=True)
X = X.fillna(med)

y = dataset["Style"].astype(str).str.strip()


In [4]:
# Scale + KMeans
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

model = KMeans(n_clusters=number_of_clusters, random_state=random_state, n_init=10)
labels = model.fit_predict(X_scaled)


In [5]:
# Majority style per cluster + cluster accuracy
cluster_style = {}
acc_rows = []

for c in range(number_of_clusters):
    idx = (labels == c)
    styles = y[idx]
    if len(styles) == 0:
        maj = ""
        size = 0
        acc = np.nan
    else:
        maj = styles.value_counts().idxmax()
        size = int(idx.sum())
        acc = float((styles == maj).sum() / size)
    cluster_style[c] = maj
    acc_rows.append({"ClusterStyle": maj, "SizeOfCluster": size, "Accuracy": acc})

cluster_cars = dataset[["Volume","Doors","Style"]].copy()
cluster_cars["ClusterStyle"] = [cluster_style[c] for c in labels]
cluster_cars.to_csv("ClusterCars.csv", index=False)

pd.DataFrame(acc_rows, columns=["ClusterStyle","SizeOfCluster","Accuracy"]).to_csv("ClusterAccuracy.csv", index=False)

cluster_cars.head(), pd.DataFrame(acc_rows)


(   Volume  Doors  Style ClusterStyle
 0     102      4  Sedan        Sedan
 1     121      5    SUV          SUV
 2     113      4  Sedan        Sedan
 3     134      5    SUV          SUV
 4     134      5    SUV          SUV,
   ClusterStyle  SizeOfCluster  Accuracy
 0          SUV              9  0.555556
 1          SUV             30  0.733333
 2        Sedan             83  0.578313
 3        Sedan              7  0.714286
 4          SUV             31  0.580645)